# NoSQL on AWS 

## DocumentDB, DynamoDB, and Keyspaces

You've gone over the problems with self hosting a NoSQL database - but how can you start hosting your database on the cloud? Let's work through it together. In this exercise, you will learn how to:

1. Use the AWS Boto3 library to launch and interact with three different NoSQL database services on AWS
2. Explore **Amazon DocumentDB** (MongoDB-compatible document database)
3. Explore **Amazon DynamoDB** (fully managed key-value and document database)
4. Explore **Amazon Keyspaces** (Apache Cassandra-compatible database)
5. Clean up resources to avoid unnecessary charges

Each database has different strengths and use cases. By exploring each one, you'll understand the trade-offs between consistency, scalability, and query flexibility.

## Prerequisites

You will need:

- Your Cloud Resources AWS resources with associated Access Key ID, Secret Access Key, and Session Token
- AWS Boto3 installed and configured
- A default VPC in your AWS account

These can be found in the workspace under the tab "Cloud Resources".

## Cost Warning

**IMPORTANT:** This exercise creates AWS resources that may incur charges if left running. We will shut everything down at the end, but please monitor your AWS console to ensure all resources are deleted.

## Step 1: Install and Configure boto3

First, let's import os, and boto3, the AWS SDK for Python. Make sure you have AWS credentials configured (via the "Cloud Resources" tab in the workspace).

Copy these somewhere safe, as you will need to paste them into the code segment below. This will be your first task.

In [ ]:
%%capture
import boto3
import json
import time
import ssl
from botocore.exceptions import ClientError

## Configure AWS Credentials

#### Task 1: In the cell below, replace the placeholder values with your actual AWS credentials:

**Required:**
- `aws_access_key_id`: Your AWS Access Key ID
- `aws_secret_access_key`: Your AWS Secret Access Key
- `aws_session_token`: Your AWS Session Token

Note, it's usually not best practices to put passwords and access keys and the like into code. Hardcoding is not secure at all - this is only for demonstration purposes!

In [ ]:
import os
import boto3

# Configure AWS credentials - Replace with your actual credentials
session = boto3.Session(
    aws_access_key_id= '### BEGIN SOLUTION We cannot input this for you! ### END SOLUTION',
    aws_secret_access_key= '### BEGIN SOLUTION We cannot input this for you! ### END SOLUTION',
    aws_session_token= '### BEGIN SOLUTION We cannot input this for you! ### END SOLUTION',  # Optional: only needed for temporary credentials
    region_name='us-east-1'  # Change to your preferred region
)

# Create clients using the configured session
sts = session.client('sts')
boto3.setup_default_session(
    aws_access_key_id=session.get_credentials().access_key,
    aws_secret_access_key=session.get_credentials().secret_key,
    aws_session_token=session.get_credentials().token,
    region_name=session.region_name
)

print("Credentials configured! Run the next cell to verify.")

You should see a statement: 'Credentials configured! Run the next cell to verify.'

You can go ahead and run the next cell.

In [ ]:
# Verify AWS configuration
try:
    identity = sts.get_caller_identity()
    # Strip email from ARN and UserId if present
    arn = identity['Arn'].split('=')[0] if '=' in identity['Arn'] else identity['Arn']
    user_id = identity['UserId'].split('=')[0] if '=' in identity['UserId'] else identity['UserId']
    print("✓ AWS Configuration Verified!")
    print(f"Account: {identity['Account']}")
    print(f"User ARN: {arn}")
    print(f"User ID: {user_id}")
except ClientError as e:
    print(f"✗ Error verifying AWS credentials: {e}")
    print("\nPlease check that you've entered valid credentials in the previous cell.")

You should see something like:

✓ AWS Configuration Verified!
Account: (some random account number)

User ARN: arn:aws:sts::(same random account number as above):assumed-role/voclabs/........................

User ID: (random string)

If that's the output you're getting, you're clear to head to the next step.

## Step 2: Setup Network Infrastructure

For DocumentDB and other databases, we need to set up VPC networking. This is the same pattern we used for RDS in Lesson 2 — VPCs, security groups, and subnet groups work identically regardless of database type.

### Create Security Group

In [ ]:
# Create AWS clients
ec2 = boto3.client('ec2')
docdb = boto3.client('docdb')
dynamodb = boto3.client('dynamodb')
keyspaces = boto3.client('keyspaces')

# Get default VPC
vpcs = ec2.describe_vpcs(Filters=[{'Name': 'isDefault', 'Values': ['true']}])
vpc_id = vpcs['Vpcs'][0]['VpcId']
print(f"Using default VPC: {vpc_id}")

# Create security group for databases
sg_name = 'nosql-exercise-sg'
try:
    sg = ec2.create_security_group(
        GroupName=sg_name,
        Description='Security group for NoSQL databases exercise',
        VpcId=vpc_id
    )
    sg_id = sg['GroupId']
    print(f"Security Group created: {sg_id}")
except ClientError as e:
    if 'InvalidGroup.Duplicate' in str(e):
        sgs = ec2.describe_security_groups(GroupNames=[sg_name])
        sg_id = sgs['SecurityGroups'][0]['GroupId']
        print(f"Security Group '{sg_name}' already exists: {sg_id}")
    else:
        raise e

# Allow inbound traffic (DocumentDB port 27017, Keyspaces port 9042)
try:
    ec2.authorize_security_group_ingress(
        GroupId=sg_id,
        IpPermissions=[
            {
                'IpProtocol': 'tcp',
                'FromPort': 27017,
                'ToPort': 27017,
                'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': 'DocumentDB access'}]
            },
            {
                'IpProtocol': 'tcp',
                'FromPort': 9042,
                'ToPort': 9042,
                'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': 'Keyspaces access'}]
            }
        ]
    )
    print("Inbound rules added: Allow ports 27017 and 9042 from anywhere")
except ClientError as e:
    if 'InvalidPermission.Duplicate' in str(e):
        print("Inbound rules already exist")
    else:
        raise e

### Create DB Subnet Group

DocumentDB requires a DB subnet group. Let's get the available subnets and create one.

In [ ]:
# Get subnets in the default VPC
subnets = ec2.describe_subnets(Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}])
subnet_ids = [subnet['SubnetId'] for subnet in subnets['Subnets'][:2]]  # Use first 2 subnets
print(f"Available subnets: {subnet_ids}")

# Create DB subnet group for DocumentDB
subnet_group_name = 'nosql-exercise-subnet-group'
try:
    docdb.create_db_subnet_group(
        DBSubnetGroupName=subnet_group_name,
        DBSubnetGroupDescription='Subnet group for NoSQL exercise',
        SubnetIds=subnet_ids,
        Tags=[{'Key': 'Exercise', 'Value': 'L3-Ex3'}]
    )
    print(f"DB Subnet Group '{subnet_group_name}' created successfully!")
except ClientError as e:
    if 'DBSubnetGroupAlreadyExists' in str(e):
        print(f"DB Subnet Group '{subnet_group_name}' already exists. Will use existing group.")
    else:
        raise e

---

## Part A: Amazon DocumentDB (MongoDB-Compatible)

### What is DocumentDB?
- **MongoDB-compatible** document database
- Stores data as **JSON-like documents**
- Strong **ACID transaction support**
- **Relational query capabilities** with $lookup
- Best for: Applications needing document flexibility with relational queries

### Create DocumentDB Cluster

### Store DocumentDB Credentials in AWS Secrets Manager

We'll generate a secure password and store it in AWS Secrets Manager — **never hardcode passwords in code!**

In [ ]:
import random
import string

# Generate a random password for DocumentDB
def generate_password(length=16):
    # DocumentDB doesn't allow: / @ " (space)
    characters = string.ascii_letters + string.digits + "!#$%^&*()-_=+"
    return ''.join(random.choice(characters) for i in range(length))

# Database credentials
db_username = "docdbadmin"
db_password = generate_password()

# Create a secret structure
secret_dict = {
    "username": db_username,
    "password": db_password
}

# Store in AWS Secrets Manager
secret_name = "docdb-nosql-credentials"
secretsmanager = boto3.client('secretsmanager')

try:
    response = secretsmanager.create_secret(
        Name=secret_name,
        Description="DocumentDB credentials for L3 Exercise 3",
        SecretString=json.dumps(secret_dict)
    )
    print("Secret created successfully!")
    print(f"Secret ARN: {response['ARN']}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceExistsException':
        print(f"Secret '{secret_name}' already exists. Updating it instead...")
        response = secretsmanager.update_secret(
            SecretId=secret_name,
            SecretString=json.dumps(secret_dict)
        )
        print("Secret updated successfully!")
    else:
        raise e

def get_secret(secret_name):
    """Retrieve secret from AWS Secrets Manager"""
    try:
        response = secretsmanager.get_secret_value(SecretId=secret_name)
        return json.loads(response['SecretString'])
    except ClientError as e:
        print(f"Error retrieving secret: {e}")
        return None

print(f"\nCredentials stored securely. Username: {db_username}")
print("Password is stored in Secrets Manager — not visible in code!")

In [ ]:
cluster_id = 'nosql-docdb-cluster'

# Retrieve credentials from Secrets Manager (never hardcode passwords!)
creds = get_secret(secret_name)
master_username = creds['username']
master_password = creds['password']

print("Creating DocumentDB cluster... This may take 5-10 minutes.")

try:
    response = docdb.create_db_cluster(
        DBClusterIdentifier=cluster_id,
        Engine='docdb',
        MasterUsername=master_username,
        MasterUserPassword=master_password,
        DBSubnetGroupName=subnet_group_name,
        VpcSecurityGroupIds=[sg_id],
        BackupRetentionPeriod=1,
        PreferredBackupWindow='03:00-04:00',
        PreferredMaintenanceWindow='mon:04:00-mon:05:00',
        StorageEncrypted=False,  # Disable encryption for testing
        Tags=[{'Key': 'Exercise', 'Value': 'L3-Ex3'}]
    )
    print(f"DocumentDB cluster creation initiated: {cluster_id}")
    print(f"Status: {response['DBCluster']['Status']}")

except ClientError as e:
    if 'DBClusterAlreadyExistsFault' in str(e):
        print(f"DocumentDB cluster already exists: {cluster_id}")
    else:
        raise e

# Store connection info
DOCDB_CLUSTER_ID = cluster_id
DOCDB_SECRET_NAME = secret_name

### Create DocumentDB Instance

A cluster needs at least one instance to run.

In [ ]:
instance_id = 'nosql-docdb-instance-1'

print("Creating DocumentDB instance in the cluster...")

try:
    response = docdb.create_db_instance(
        DBInstanceIdentifier=instance_id,
        DBInstanceClass='db.t3.medium',  # db.t3.medium is the smallest DocumentDB instance class
        Engine='docdb',
        DBClusterIdentifier=cluster_id,
        Tags=[{'Key': 'Exercise', 'Value': 'L3-Ex3'}]
    )
    print(f"DocumentDB instance creation initiated: {instance_id}")
    print(f"Instance Class: db.t3.medium")
    print(f"Status: {response['DBInstance']['DBInstanceStatus']}")

except ClientError as e:
    if 'DBInstanceAlreadyExists' in str(e):
        print(f"DocumentDB instance already exists: {instance_id}")
    else:
        raise e

# Wait for the DocumentDB instance to become available
print("\nWaiting for DocumentDB instance to become available (this typically takes 5-10 minutes)...")
try:
    waiter = docdb.get_waiter('db_instance_available')
    waiter.wait(
        DBInstanceIdentifier=instance_id,
        WaiterConfig={'Delay': 30, 'MaxAttempts': 40}  # up to ~20 minutes
    )
    print("✓ DocumentDB instance is now AVAILABLE!")
except Exception as e:
    print(f"⚠ Waiter finished with: {e}")
    print("The instance may still be starting — check status in a later cell.")

---

## Part B: Amazon DynamoDB (Key-Value/Document)

### What is DynamoDB?
- **Fully managed** key-value and document database
- **Serverless** - no infrastructure to manage
- **Single-digit millisecond latency** at any scale
- **Partition key + Sort key** design
- Best for: High-performance applications with simple query patterns

### Create DynamoDB Table

Unlike DocumentDB and Keyspaces, DynamoDB creates instantly!

In [ ]:
table_name = 'nosql-orders'

print("Creating DynamoDB table...")

try:
    response = dynamodb.create_table(
        TableName=table_name,
        KeySchema=[
            {'AttributeName': 'customer_id', 'KeyType': 'HASH'},      # Partition key
            {'AttributeName': 'order_id', 'KeyType': 'RANGE'}        # Sort key
        ],
        AttributeDefinitions=[
            {'AttributeName': 'customer_id', 'AttributeType': 'S'},  # String
            {'AttributeName': 'order_id', 'AttributeType': 'S'}      # String
        ],
        BillingMode='PAY_PER_REQUEST'  # On-demand pricing (no capacity planning)
    )
    print(f"DynamoDB table creation initiated: {table_name}")
    print(f"Key Schema: customer_id (HASH) + order_id (RANGE)")
    print(f"Billing Mode: Pay-Per-Request (Serverless)")

    # Wait for table to become active before inserting data
    print("Waiting for table to become active...")
    waiter = dynamodb.get_waiter('table_exists')
    waiter.wait(TableName=table_name)
    print("Table is now ACTIVE!")

except ClientError as e:
    if 'ResourceInUseException' in str(e):
        print(f"DynamoDB table already exists: {table_name}")
    else:
        raise e

DYNAMODB_TABLE_NAME = table_name

### Insert Data into DynamoDB

DynamoDB is already ready! Let's add some sample orders.

In [ ]:
print("Inserting sample data into DynamoDB...\n")

sample_orders = [
    {
        'customer_id': {'S': 'cust-001'},
        'order_id': {'S': 'order-101'},
        'product': {'S': 'Laptop'},
        'total': {'N': '1200.00'},
        'order_date': {'S': '2024-01-15'}
    },
    {
        'customer_id': {'S': 'cust-001'},
        'order_id': {'S': 'order-102'},
        'product': {'S': 'Mouse'},
        'total': {'N': '50.00'},
        'order_date': {'S': '2024-01-16'}
    },
    {
        'customer_id': {'S': 'cust-002'},
        'order_id': {'S': 'order-103'},
        'product': {'S': 'Keyboard'},
        'total': {'N': '75.00'},
        'order_date': {'S': '2024-01-17'}
    }
]

try:
    for order in sample_orders:
        dynamodb.put_item(TableName=table_name, Item=order)
        print(f"Inserted order: {order['order_id']['S']}")
    print(f"\nAll {len(sample_orders)} orders inserted successfully")
except ClientError as e:
    print(f"Error inserting data: {e}")

### Query DynamoDB

DynamoDB queries work on the **Partition Key + Sort Key** design. Let's get all orders for a customer.

In [ ]:
print("Querying DynamoDB: Get all orders for customer 'cust-001'\n")
print("Query Pattern: Partition Key (customer_id) + Sort Key Range (order_id)\n")

try:
    response = dynamodb.query(
        TableName=table_name,
        KeyConditionExpression='customer_id = :cid',
        ExpressionAttributeValues={
            ':cid': {'S': 'cust-001'}
        }
    )

    print(f"Found {response['Count']} orders:\n")
    for item in response['Items']:
        print(f"  Order ID: {item['order_id']['S']}")
        print(f"  Product: {item['product']['S']}")
        print(f"  Total: ${item['total']['N']}")
        print()

    print(f"DynamoDB Query Characteristics:")
    print(f"  - Query by partition key + sort key only")
    print(f"  - No complex joins or aggregations")
    print(f"  - Ultra-fast single-digit millisecond response")
    print(f"  - Predictable, scalable performance")

except ClientError as e:
    print(f"Error querying table: {e}")

---

## Part C: Amazon Keyspaces (Apache Cassandra-Compatible)

### What is Keyspaces?
- **Apache Cassandra-compatible** database
- **Distributed**, **highly available** architecture
- **Partition + clustering keys** for time-series and IoT data
- **Eventual consistency** for high availability
- Best for: Time-series data, IoT, massive scale with high availability

### Create Keyspace and Table

In [ ]:
keyspace_name = 'nosql_exercise'
table_name_keyspaces = 'orders'

print("Creating Keyspaces (Cassandra) keyspace and table...\n")

try:
    # Create keyspace
    response = keyspaces.create_keyspace(
        keyspaceName=keyspace_name,
        tags=[{'key': 'Exercise', 'value': 'L3-Ex3'}]
    )
    print(f"Created keyspace: {keyspace_name}")
    print(f"ARN: {response['resourceArn']}")

except ClientError as e:
    if 'ConflictException' in str(e):
        print(f"Keyspace already exists: {keyspace_name}")
    else:
        raise e

# Wait for keyspace to become ACTIVE
# If get_keyspace succeeds without error, the keyspace is ready.
print("Waiting for keyspace to become ACTIVE...")
for _ in range(30):
    try:
        keyspaces.get_keyspace(keyspaceName=keyspace_name)
        print(f"Keyspace '{keyspace_name}' is ready!")
        break
    except ClientError:
        time.sleep(2)

print("\nCreating table in keyspace...\n")

try:
    # Create table
    response = keyspaces.create_table(
        keyspaceName=keyspace_name,
        tableName=table_name_keyspaces,
        schemaDefinition={
            'allColumns': [
                {'name': 'customer_id', 'type': 'text'},
                {'name': 'order_date', 'type': 'timestamp'},
                {'name': 'order_id', 'type': 'text'},
                {'name': 'product', 'type': 'text'},
                {'name': 'total', 'type': 'decimal'}
            ],
            'partitionKeys': [
                {'name': 'customer_id'}
            ],
            'clusteringKeys': [
                {'name': 'order_date', 'orderBy': 'DESC'},
                {'name': 'order_id', 'orderBy': 'ASC'}
            ]
        },
        capacitySpecification={'throughputMode': 'PAY_PER_REQUEST'},
        tags=[{'key': 'Exercise', 'value': 'L3-Ex3'}]
    )
    print(f"Created Keyspaces table: {table_name_keyspaces}")
    print(f"Partition Key: customer_id")
    print(f"Clustering Keys: order_date (DESC), order_id (ASC)")

except ClientError as e:
    if 'ConflictException' in str(e):
        print(f"Table already exists: {table_name_keyspaces}")
    else:
        raise e

# Poll until the table is ACTIVE (instead of a blind sleep)
print("\nWaiting for Keyspaces table to become ACTIVE...")
for i in range(60):  # up to ~2 minutes
    try:
        tbl = keyspaces.get_table(keyspaceName=keyspace_name, tableName=table_name_keyspaces)
        status = tbl.get('status', 'UNKNOWN')
        if status == 'ACTIVE':
            print(f"✓ Keyspaces table '{table_name_keyspaces}' is ACTIVE!")
            break
        print(f"  Table status: {status} — checking again in 2s...")
    except ClientError:
        print(f"  Table not ready yet — checking again in 2s...")
    time.sleep(2)
else:
    print("⚠ Timed out waiting for table. It may still be initializing — continue and retry later.")

KEYSPACES_KEYSPACE = keyspace_name
KEYSPACES_TABLE = table_name_keyspaces

### Insert Data into Keyspaces

We'll use the Cassandra Python driver to interact with Keyspaces.

In [ ]:
from cassandra.cluster import Cluster
from cassandra.policies import DCAwareRoundRobinPolicy
from cassandra.query import ConsistencyLevel
from cassandra_sigv4.auth import SigV4AuthProvider
from datetime import datetime
from decimal import Decimal

print("Connecting to Amazon Keyspaces via SigV4 authentication...\n")

try:
    # SigV4 authentication using our existing AWS session credentials
    auth_provider = SigV4AuthProvider(session=session)

    # TLS is required for Amazon Keyspaces
    # Disable hostname checking because the Cassandra driver resolves the
    # contact-point hostname to IP addresses and connects to those IPs directly.
    # The Keyspaces certificate is issued for the hostname, not the IPs,
    # which causes an "IP address mismatch" error with default settings.
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False

    keyspaces_cluster = Cluster(
        contact_points=['cassandra.us-east-1.amazonaws.com'],
        port=9142,  # TLS port for Keyspaces
        auth_provider=auth_provider,
        ssl_context=ssl_context,
        load_balancing_policy=DCAwareRoundRobinPolicy(local_dc='us-east-1')
    )

    cassandra_session = keyspaces_cluster.connect()

    # Amazon Keyspaces only supports LOCAL_QUORUM consistency.
    # The driver defaults to LOCAL_ONE, which Keyspaces rejects.
    cassandra_session.default_consistency_level = ConsistencyLevel.LOCAL_QUORUM

    print("Connected to Keyspaces!")

    # Insert sample data
    insert_query = f"""
    INSERT INTO {KEYSPACES_KEYSPACE}.{KEYSPACES_TABLE}
    (customer_id, order_date, order_id, product, total)
    VALUES (?, ?, ?, ?, ?)
    """

    prepared = cassandra_session.prepare(insert_query)

    orders_data = [
        ('cust-001', datetime(2024, 1, 15), 'order-101', 'Laptop', Decimal('1200.00')),
        ('cust-001', datetime(2024, 1, 16), 'order-102', 'Mouse', Decimal('50.00')),
        ('cust-002', datetime(2024, 1, 17), 'order-103', 'Keyboard', Decimal('75.00'))
    ]

    for order in orders_data:
        cassandra_session.execute(prepared, order)
        print(f"Inserted order: {order[2]}")

    print(f"\nAll {len(orders_data)} orders inserted")

except ImportError:
    print("Cassandra driver not installed. Install with: pip install cassandra-driver cassandra-sigv4")
    print("\nContinuing without inserting data...\n")
except Exception as e:
    print(f"Connection to Keyspaces: {e}")
    print("This is expected if Keyspaces table is still initializing (takes ~1-2 minutes)")

### Query Keyspaces

Keyspaces excels at time-series queries with clustering keys.

In [ ]:
print("Querying Keyspaces: Get orders for customer in date range\n")
print("Query Pattern: Partition Key (customer_id) + Clustering Keys (order_date DESC, order_id ASC)\n")

try:
    query = f"""
    SELECT customer_id, order_date, order_id, product, total
    FROM {KEYSPACES_KEYSPACE}.{KEYSPACES_TABLE}
    WHERE customer_id = 'cust-001'
    ORDER BY order_date DESC
    """

    results = cassandra_session.execute(query)

    print("Results:\n")
    count = 0
    for row in results:
        print(f"  Order ID: {row.order_id}")
        print(f"  Date: {row.order_date}")
        print(f"  Product: {row.product}")
        print(f"  Total: ${row.total}")
        print()
        count += 1

    print(f"Keyspaces Query Characteristics:")
    print(f"  - Optimized for time-series data")
    print(f"  - Clustering keys enable efficient range queries")
    print(f"  - Distributed, highly available")
    print(f"  - Eventually consistent")

except Exception as e:
    print(f"Query failed: {e}")

---

## Comparison: DocumentDB vs DynamoDB vs Keyspaces

Let's summarize the key differences we've seen:

In [ ]:
import pandas as pd

comparison_data = {
    'Feature': [
        'Data Model',
        'Query Language',
        'Key Design',
        'Consistency',
        'Best For',
        'Joins',
        'Aggregations',
        'Time to Create'
    ],
    'DocumentDB': [
        'Document (MongoDB-like)',
        'MongoDB Query Language',
        'No key requirement',
        'Strong (ACID)',
        'Relational queries, complex documents',
        '$lookup (slower)',
        'Yes (aggregation pipeline)',
        '5-10 minutes'
    ],
    'DynamoDB': [
        'Key-Value/Document',
        'DynamoDB API',
        'Partition + Sort Key',
        'Strong (within partition)',
        'High-performance simple queries',
        'Not supported',
        'Not supported',
        'Instant'
    ],
    'Keyspaces (Cassandra)': [
        'Column-oriented',
        'CQL (Cassandra Query Language)',
        'Partition + Clustering Keys',
        'Eventual (tunable)',
        'Time-series, IoT, massive scale',
        'Not supported',
        'Limited',
        '1-2 minutes'
    ]
}

df = pd.DataFrame(comparison_data)
print("\n" + "="*120)
print("COMPARISON: NoSQL Databases on AWS")
print("="*120 + "\n")
print(df.to_string(index=False))
print("\n" + "="*120)

---

## Check DocumentDB Status

Let's check if our DocumentDB cluster is ready yet. (Creation usually takes 5-10 minutes)

In [ ]:
print("Checking DocumentDB cluster status...\n")

try:
    response = docdb.describe_db_clusters(DBClusterIdentifier=DOCDB_CLUSTER_ID)
    cluster = response['DBClusters'][0]

    print(f"Cluster: {cluster['DBClusterIdentifier']}")
    print(f"Status: {cluster['Status']}")
    print(f"Endpoint: {cluster.get('Endpoint', 'Not yet available')}")
    print(f"Port: {cluster.get('Port', 27017)}")

    if cluster['Status'] == 'available':
        # Retrieve username from Secrets Manager
        creds = get_secret(DOCDB_SECRET_NAME)
        print(f"\nDocumentDB is ready! The connection string would be:")
        print(f"mongodb://{creds['username']}:<password>@{cluster['Endpoint']}:27017/?tls=true&tlsCAFile=global-bundle.pem")
        print(f"(Password is stored in Secrets Manager secret: '{DOCDB_SECRET_NAME}')")
        print(f"\nNote: DocumentDB is VPC-only — you'd need a bastion host or VPN to connect from outside the VPC.")
    else:
        print(f"\nDocumentDB is still {cluster['Status'].lower()}. Check again in a few minutes.")

except ClientError as e:
    print(f"Error checking status: {e}")

## Query anything!

#### Task 2: Query anything you want across each of the databases!

In [ ]:
### BEGIN SOLUTION

# ── DynamoDB: Query all orders for customer cust-002 ──
print("DynamoDB — Orders for cust-002:")
print("-" * 40)
try:
    resp = dynamodb.query(
        TableName=DYNAMODB_TABLE_NAME,
        KeyConditionExpression='customer_id = :cid',
        ExpressionAttributeValues={':cid': {'S': 'cust-002'}}
    )
    for item in resp['Items']:
        print(f"  {item['order_id']['S']}: {item['product']['S']} — ${item['total']['N']}")
    if resp['Count'] == 0:
        print("  (no results)")
except ClientError as e:
    print(f"  Error: {e}")

# ── DynamoDB: Scan for all orders over $100 ──
print("\nDynamoDB — All orders over $100:")
print("-" * 40)
try:
    resp = dynamodb.scan(
        TableName=DYNAMODB_TABLE_NAME,
        FilterExpression='#t > :min',
        ExpressionAttributeNames={'#t': 'total'},
        ExpressionAttributeValues={':min': {'N': '100'}}
    )
    for item in resp['Items']:
        print(f"  {item['customer_id']['S']} / {item['order_id']['S']}: {item['product']['S']} — ${item['total']['N']}")
    if resp['Count'] == 0:
        print("  (no results)")
except ClientError as e:
    print(f"  Error: {e}")

# ── Keyspaces: Query the most recent order for cust-001 ──
print("\nKeyspaces — Most recent order for cust-001:")
print("-" * 40)
try:
    rows = cassandra_session.execute(f"""
        SELECT order_id, order_date, product, total
        FROM {KEYSPACES_KEYSPACE}.{KEYSPACES_TABLE}
        WHERE customer_id = 'cust-001'
        ORDER BY order_date DESC
        LIMIT 1
    """)
    for row in rows:
        print(f"  {row.order_id}: {row.product} — ${row.total} ({row.order_date})")
except Exception as e:
    print(f"  Skipped (Cassandra driver not connected): {e}")

# ── DocumentDB: Check if cluster is available and show connection info ──
print("\nDocumentDB — Cluster status:")
print("-" * 40)
try:
    cluster = docdb.describe_db_clusters(
        DBClusterIdentifier=DOCDB_CLUSTER_ID
    )['DBClusters'][0]
    print(f"  Status: {cluster['Status']}")
    if cluster['Status'] == 'available':
        print(f"  Endpoint: {cluster['Endpoint']}:{cluster['Port']}")
        print(f"  (DocumentDB requires VPC connectivity or a bastion host to query directly)")
    else:
        print(f"  Still provisioning — try again in a few minutes.")
except ClientError as e:
    print(f"  Error: {e}")

### END SOLUTION

---

## Clean Up Resources

**IMPORTANT:** To avoid unnecessary AWS charges, make sure to clean up the resources you created in this exercise.

In [ ]:
print("CLEANUP: Deleting all AWS resources created in this exercise")
print("="*80 + "\n")

# ── Step 1: Delete DynamoDB table ──
print("Step 1: Deleting DynamoDB table...")
try:
    dynamodb.delete_table(TableName=DYNAMODB_TABLE_NAME)
    print(f"  Deletion initiated: '{DYNAMODB_TABLE_NAME}'")
    # Wait until the table no longer exists
    print("  Waiting for table deletion...")
    waiter = dynamodb.get_waiter('table_not_exists')
    waiter.wait(TableName=DYNAMODB_TABLE_NAME, WaiterConfig={'Delay': 5, 'MaxAttempts': 30})
    print(f"  ✓ DynamoDB table '{DYNAMODB_TABLE_NAME}' deleted.")
except ClientError as e:
    if 'ResourceNotFoundException' in str(e):
        print(f"  Table already deleted: {DYNAMODB_TABLE_NAME}")
    else:
        print(f"  Error: {e}")
except Exception as e:
    print(f"  Waiter error (table may still be deleting): {e}")

print()

# ── Step 2: Delete Keyspaces table ──
print("Step 2: Deleting Keyspaces table...")
try:
    keyspaces.delete_table(
        keyspaceName=KEYSPACES_KEYSPACE,
        tableName=KEYSPACES_TABLE
    )
    print(f"  Deletion initiated: '{KEYSPACES_TABLE}'")
    # Poll until the table is gone
    print("  Waiting for Keyspaces table deletion...")
    for _ in range(60):
        try:
            tbl = keyspaces.get_table(keyspaceName=KEYSPACES_KEYSPACE, tableName=KEYSPACES_TABLE)
            status = tbl.get('status', 'UNKNOWN')
            if status == 'ACTIVE':
                # Still there — keep waiting
                time.sleep(3)
                continue
            # Status might be DELETING — keep waiting
            time.sleep(3)
        except ClientError as inner_e:
            if 'ResourceNotFoundException' in str(inner_e):
                print(f"  ✓ Keyspaces table '{KEYSPACES_TABLE}' deleted.")
                break
            time.sleep(3)
    else:
        print("  ⚠ Timed out — table may still be deleting.")
except ClientError as e:
    if 'ResourceNotFoundException' in str(e):
        print(f"  Table already deleted: {KEYSPACES_TABLE}")
    else:
        print(f"  Error: {e}")

print()

# ── Step 3: Delete Keyspaces keyspace ──
print("Step 3: Deleting Keyspaces keyspace...")
try:
    keyspaces.delete_keyspace(keyspaceName=KEYSPACES_KEYSPACE)
    print(f"  Deletion initiated: '{KEYSPACES_KEYSPACE}'")
    # Poll until the keyspace is gone
    print("  Waiting for keyspace deletion...")
    for _ in range(60):
        try:
            keyspaces.get_keyspace(keyspaceName=KEYSPACES_KEYSPACE)
            time.sleep(3)
        except ClientError as inner_e:
            if 'ResourceNotFoundException' in str(inner_e):
                print(f"  ✓ Keyspace '{KEYSPACES_KEYSPACE}' deleted.")
                break
            time.sleep(3)
    else:
        print("  ⚠ Timed out — keyspace may still be deleting.")
except ClientError as e:
    if 'ResourceNotFoundException' in str(e):
        print(f"  Keyspace already deleted: {KEYSPACES_KEYSPACE}")
    else:
        print(f"  Error: {e}")

print()

# ── Step 4: Delete DocumentDB instance ──
print("Step 4: Deleting DocumentDB instance...")
try:
    docdb.delete_db_instance(DBInstanceIdentifier=instance_id)
    print(f"  Deletion initiated: '{instance_id}'")
    # Wait until the instance is fully deleted before touching the cluster
    print("  Waiting for instance deletion (this may take several minutes)...")
    waiter = docdb.get_waiter('db_instance_deleted')
    waiter.wait(
        DBInstanceIdentifier=instance_id,
        WaiterConfig={'Delay': 30, 'MaxAttempts': 40}
    )
    print(f"  ✓ DocumentDB instance '{instance_id}' deleted.")
except ClientError as e:
    if 'DBInstanceNotFound' in str(e):
        print(f"  Instance already deleted: {instance_id}")
    else:
        print(f"  Error: {e}")
except Exception as e:
    print(f"  Waiter error (instance may still be deleting): {e}")

print()

# ── Step 5: Delete DocumentDB cluster ──
print("Step 5: Deleting DocumentDB cluster...")
try:
    docdb.delete_db_cluster(
        DBClusterIdentifier=DOCDB_CLUSTER_ID,
        SkipFinalSnapshot=True
    )
    print(f"  Deletion initiated: '{DOCDB_CLUSTER_ID}'")
    # Poll until the cluster is gone
    print("  Waiting for cluster deletion...")
    for _ in range(60):
        try:
            resp = docdb.describe_db_clusters(DBClusterIdentifier=DOCDB_CLUSTER_ID)
            status = resp['DBClusters'][0]['Status']
            print(f"    Cluster status: {status}")
            time.sleep(15)
        except ClientError as inner_e:
            if 'DBClusterNotFoundFault' in str(inner_e):
                print(f"  ✓ DocumentDB cluster '{DOCDB_CLUSTER_ID}' deleted.")
                break
            time.sleep(15)
    else:
        print("  ⚠ Timed out — cluster may still be deleting.")
except ClientError as e:
    if 'DBClusterNotFoundFault' in str(e):
        print(f"  Cluster already deleted: {DOCDB_CLUSTER_ID}")
    else:
        print(f"  Error: {e}")

print()

# ── Step 6: Delete Secrets Manager secret ──
print("Step 6: Deleting Secrets Manager secret...")
try:
    secretsmanager.delete_secret(
        SecretId=DOCDB_SECRET_NAME,
        ForceDeleteWithoutRecovery=True
    )
    print(f"  ✓ Secret '{DOCDB_SECRET_NAME}' deleted.")
except ClientError as e:
    if 'ResourceNotFoundException' in str(e):
        print(f"  Secret already deleted: {DOCDB_SECRET_NAME}")
    else:
        print(f"  Error: {e}")

print(f"\n{'='*80}")
print("Steps 1-6 complete. Run the next cell to clean up network resources.")

### Clean Up Network Resources

Delete the DB subnet group and security group.

In [ ]:
# ── Step 7: Delete DB subnet group ──
print("Step 7: Deleting DB subnet group...")
# Subnet group can only be deleted once all DocumentDB instances/clusters using it are gone.
# Retry with back-off in case deletion is still propagating.
for attempt in range(6):
    try:
        docdb.delete_db_subnet_group(DBSubnetGroupName=subnet_group_name)
        print(f"  ✓ DB Subnet Group '{subnet_group_name}' deleted.")
        break
    except ClientError as e:
        if 'DBSubnetGroupNotFoundFault' in str(e):
            print(f"  Subnet group already deleted.")
            break
        elif 'InvalidDBSubnetGroupState' in str(e):
            wait = 15 * (attempt + 1)
            print(f"  Subnet group still in use — waiting {wait}s before retrying...")
            time.sleep(wait)
        else:
            print(f"  Error: {e}")
            break
else:
    print("  ⚠ Could not delete subnet group after several attempts.")
    print("  Wait a few minutes and re-run this cell.")

print()

# ── Step 8: Delete security group ──
print("Step 8: Deleting security group...")
for attempt in range(6):
    try:
        ec2.delete_security_group(GroupId=sg_id)
        print(f"  ✓ Security Group '{sg_name}' deleted.")
        break
    except ClientError as e:
        if 'InvalidGroup.NotFound' in str(e):
            print(f"  Security group already deleted.")
            break
        elif 'DependencyViolation' in str(e):
            wait = 15 * (attempt + 1)
            print(f"  Security group still in use — waiting {wait}s before retrying...")
            time.sleep(wait)
        else:
            print(f"  Error: {e}")
            break
else:
    print("  ⚠ Could not delete security group after several attempts.")
    print("  Wait a few minutes and re-run this cell.")

print(f"\n{'='*80}")
print("All cleanup complete! Verify in the AWS Console that no resources remain.")

Congratulations! You now know how to launch and manage NoSQL databases on AWS using Amazon's Boto3 Python library.

## Summary

### What You Learned

1. **Amazon DocumentDB**
   - MongoDB-compatible document database
   - Strong ACID transactions with relational query support
   - Use for applications needing document flexibility with complex queries

2. **Amazon DynamoDB**
   - Fully managed, serverless key-value store
   - Single-digit millisecond latency at massive scale
   - Designed for simple, predictable query patterns
   - No joins or complex aggregations

3. **Amazon Keyspaces**
   - Apache Cassandra-compatible distributed database
   - Optimized for time-series and IoT workloads
   - Clustering keys enable efficient range queries
   - Eventually consistent for high availability

### Key Takeaways

- **Choose your database based on your query patterns**, not data format
- **DocumentDB**: Complex queries on documents
- **DynamoDB**: Simple, high-performance lookups
- **Keyspaces**: Time-series and historical data analysis
- The infrastructure patterns (VPCs, security groups, subnet groups) are **the same** regardless of database type
- **Always clean up resources** to avoid unnecessary AWS charges